In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import torch
import timm
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

In [ ]:
train_dir = "/content/drive/MyDrive/Poultry/dataset/train"
val_dir   = "/content/drive/MyDrive/Poultry/dataset/val"
test_dir  = "/content/drive/MyDrive/Poultry/dataset/test"

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds = datasets.ImageFolder(val_dir, transform=transform)
test_ds = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)


print("Number of training images   :", len(train_ds))
print("Number of validation images :", len(val_ds))
print("Number of testing images    :", len(test_ds))

# Total
print("Total images :", len(train_ds) + len(val_ds) + len(test_ds))


Number of training images   : 4260
Number of validation images : 409
Number of testing images    : 202
Total images : 4871


In [ ]:
# ============================================================
# SWIN + CONVNEXTV2 ABLATION STUDY
# A3 - A7 AUTOMATIC TRAINING
#
# A3 = Swin + ConvNeXtV2
# A4 = A3 + SE Attention
# A5 = A3 + Cross Attention
# A6 = A3 + SE + Cross Attention
# A7 = A3 + SE + Cross Attention + Gated Fusion
#
# OUTPUT:
# Exact Train Accuracy
# Exact Validation Accuracy
# Exact Test Accuracy
# Best Epoch
# ============================================================



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


BATCH_SIZE = 16
EPOCHS = 30
NUM_WORKERS = 2

NUM_CLASSES = len(train_ds.classes)

VARIANTS = [
    "A3",
    "A4",
    "A5",
    "A6",
    "A7"
]

print("Classes:", train_ds.classes)
print("Number of classes:", NUM_CLASSES)
print("Variants:", VARIANTS)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

# ============================================================
# SE ATTENTION
# ============================================================
class SEAttention(nn.Module):
    def __init__(self, dim, reduction=16):
        super().__init__()
        hidden = max(dim // reduction, 1)
        self.fc = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, dim
            ),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(x)

# ============================================================
# FEATURE PROJECTION
# ============================================================

class FeatureProjection(nn.Module):
    def __init__(self, in_dim, out_dim=384):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU()
        )

    def forward(self, x):
        return self.proj(x)


# ============================================================
# CROSS ATTENTION
# ============================================================

class CrossAttention(nn.Module):

    def __init__(self, dim=384,
        heads=8
    ):

        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            batch_first=True
        )

        self.norm = nn.LayerNorm(dim)

    def forward(
        self,
        query,
        key_value
    ):

        attended, _ = self.attention(
            query,
            key_value,
            key_value
        )

        return self.norm(
            query + attended
        )


# ============================================================
# GATED FUSION
# ============================================================

class GatedFusion(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.gate = nn.Sequential(

            nn.Linear(
                dim * 2,
                dim
            ),

            nn.GELU(),

            nn.Linear(
                dim,
                dim
            ),

            nn.Sigmoid()
        )

    def forward(
        self,
        swin,
        conv
    ):

        combined = torch.cat(
            [swin, conv],
            dim=1
        )

        gate = self.gate(
            combined
        )

        fused = (
            gate * swin
            +
            (1.0 - gate) * conv
        )

        return fused


# ============================================================
# HYBRID ABLATION MODEL
# ============================================================

class SwinConvNeXtAblation(nn.Module):

    def __init__(
        self,
        variant,
        num_classes
    ):

        super().__init__()

        self.variant = variant

        # ----------------------------------------------------
        # SWIN
        # ----------------------------------------------------

        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )

        # ----------------------------------------------------
        # CONVNEXT V2
        # ----------------------------------------------------

        self.conv = timm.create_model(
            "convnextv2_tiny.fcmae_ft_in22k_in1k",
            pretrained=True,
            num_classes=0
        )

        swin_dim = self.swin.num_features
        conv_dim = self.conv.num_features

        print(
            "Swin dimension:",
            swin_dim
        )

        print(
            "ConvNeXtV2 dimension:",
            conv_dim
        )

        # ----------------------------------------------------
        # COMMON DIMENSION
        # ----------------------------------------------------

        fusion_dim = 384

        self.swin_projection = FeatureProjection(
            swin_dim,
            fusion_dim
        )

        self.conv_projection = FeatureProjection(
            conv_dim,
            fusion_dim
        )

        # ----------------------------------------------------
        # SE
        # ----------------------------------------------------

        self.se_swin = SEAttention(
            fusion_dim
        )

        self.se_conv = SEAttention(
            fusion_dim
        )

        # ----------------------------------------------------
        # CROSS ATTENTION
        # ----------------------------------------------------

        self.cross_swin = CrossAttention(
            fusion_dim,
            heads=8
        )

        self.cross_conv = CrossAttention(
            fusion_dim,
            heads=8
        )

        # ----------------------------------------------------
        # GATED FUSION
        # ----------------------------------------------------

        self.gated_fusion = GatedFusion(
            fusion_dim
        )

        # ----------------------------------------------------
        # CLASSIFIER
        # ----------------------------------------------------

        if variant == "A7":

            classifier_input = fusion_dim

        else:

            classifier_input = fusion_dim * 2

        self.classifier = nn.Sequential(

            nn.LayerNorm(
                classifier_input
            ),

            nn.Linear(
                classifier_input,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                0.4
            ),

            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                128,
                num_classes
            )
        )


    # ========================================================
    # FORWARD
    # ========================================================

    def forward(self, x):

        # ----------------------------------------------------
        # SWIN FEATURES
        # Output usually:
        # [B, 7, 7, 768]
        # ----------------------------------------------------

        swin_features = (
            self.swin.forward_features(x)
        )

        # ----------------------------------------------------
        # CONVNEXT FEATURES
        # Output usually:
        # [B, 768, 7, 7]
        # ----------------------------------------------------

        conv_features = (
            self.conv.forward_features(x)
        )

        # ----------------------------------------------------
        # SWIN -> [B, 49, C]
        # ----------------------------------------------------

        if swin_features.ndim == 4:

            # Swin: B,H,W,C

            swin_tokens = swin_features.reshape(
                swin_features.shape[0],
                -1,
                swin_features.shape[-1]
            )

        else:

            swin_tokens = swin_features


        # ----------------------------------------------------
        # CONVNEXT -> [B, 49, C]
        # ----------------------------------------------------

        if conv_features.ndim == 4:

            # ConvNeXt: B,C,H,W

            conv_tokens = conv_features.permute(
                0,
                2,
                3,
                1
            )

            conv_tokens = conv_tokens.reshape(
                conv_tokens.shape[0],
                -1,
                conv_tokens.shape[-1]
            )

        else:

            conv_tokens = conv_features


        # ----------------------------------------------------
        # PROJECT TO COMMON DIMENSION
        # ----------------------------------------------------

        swin_tokens = self.swin_projection(
            swin_tokens
        )

        conv_tokens = self.conv_projection(
            conv_tokens
        )


        # ====================================================
        # A3
        # SIMPLE FEATURE FUSION
        # ====================================================

        if self.variant == "A3":

            swin_vec = swin_tokens.mean(
                dim=1
            )

            conv_vec = conv_tokens.mean(
                dim=1
            )

            fused = torch.cat(
                [
                    swin_vec,
                    conv_vec
                ],
                dim=1
            )

            return self.classifier(
                fused
            )


        # ====================================================
        # A4
        # SE / ECA
        # ====================================================

        if self.variant == "A4":

            swin_vec = swin_tokens.mean(
                dim=1
            )

            conv_vec = conv_tokens.mean(
                dim=1
            )

            swin_vec = self.se_swin(
                swin_vec
            )

            conv_vec = self.se_conv(
                conv_vec
            )

            fused = torch.cat(
                [
                    swin_vec,
                    conv_vec
                ],
                dim=1
            )

            return self.classifier(
                fused
            )


        # ====================================================
        # A5
        # CROSS ATTENTION
        # ====================================================

        if self.variant == "A5":

            swin_attended = self.cross_swin(
                swin_tokens,
                conv_tokens
            )

            conv_attended = self.cross_conv(
                conv_tokens,
                swin_tokens
            )

            swin_vec = swin_attended.mean(
                dim=1
            )

            conv_vec = conv_attended.mean(
                dim=1
            )

            fused = torch.cat(
                [
                    swin_vec,
                    conv_vec
                ],
                dim=1
            )

            return self.classifier(
                fused
            )


        # ====================================================
        # A6
        # CROSS ATTENTION + SE
        # ====================================================

        if self.variant == "A6":

            swin_attended = self.cross_swin(
                swin_tokens,
                conv_tokens
            )

            conv_attended = self.cross_conv(
                conv_tokens,
                swin_tokens
            )

            swin_vec = swin_attended.mean(
                dim=1
            )

            conv_vec = conv_attended.mean(
                dim=1
            )

            swin_vec = self.se_swin(
                swin_vec
            )

            conv_vec = self.se_conv(
                conv_vec
            )

            fused = torch.cat(
                [
                    swin_vec,
                    conv_vec
                ],
                dim=1
            )

            return self.classifier(
                fused
            )


        # ====================================================
        # A7
        # CROSS ATTENTION + SE + GATED FUSION
        # ====================================================

        if self.variant == "A7":

            swin_attended = self.cross_swin(
                swin_tokens,
                conv_tokens
            )

            conv_attended = self.cross_conv(
                conv_tokens,
                swin_tokens
            )

            swin_vec = swin_attended.mean(
                dim=1
            )

            conv_vec = conv_attended.mean(
                dim=1
            )

            swin_vec = self.se_swin(
                swin_vec
            )

            conv_vec = self.se_conv(
                conv_vec
            )

            fused = self.gated_fusion(
                swin_vec,
                conv_vec
            )

            return self.classifier(
                fused
            )


        raise ValueError(
            "Unknown variant"
        )


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_accuracy(
    model,
    loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                "cuda",
                enabled=(device.type == "cuda")
            ):

                outputs = model(
                    images
                )

            predictions = outputs.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return correct / total


# ============================================================
# TRAIN ONE VARIANT
# ============================================================

def train_variant(
    variant
):

    print("\n")
    print("=" * 70)
    print(f"STARTING ABLATION {variant}")
    print("=" * 70)


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = SwinConvNeXtAblation(
        variant,
        NUM_CLASSES
    ).to(device)


    # --------------------------------------------------------
    # LOSS
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )


    # --------------------------------------------------------
    # DIFFERENTIAL LR
    # --------------------------------------------------------

    backbone_params = []
    new_params = []

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if any(
            key in name
            for key in [
                "projection",
                "se_",
                "cross_",
                "gated_fusion",
                "classifier"
            ]
        ):

            new_params.append(
                param
            )

        else:

            backbone_params.append(
                param
            )


    optimizer = torch.optim.AdamW(

        [
            {
                "params": backbone_params,
                "lr": 3e-5
            },

            {
                "params": new_params,
                "lr": 1e-4
            }
        ],

        weight_decay=1e-4
    )


    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-6
    )


    # --------------------------------------------------------
    # AMP
    # --------------------------------------------------------

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda")
    )


    # --------------------------------------------------------
    # BEST
    # --------------------------------------------------------

    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_epoch = 0

    best_train_acc = 0.0

    patience = 7
    patience_counter = 0

    save_path = (
        f"ablation_{variant}_best.pth"
    )


    # ========================================================
    # TRAINING LOOP
    # ========================================================

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0.0

        correct = 0
        total = 0


        # ====================================================
        # TRAIN
        # ====================================================

        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.amp.autocast(
                "cuda",
                enabled=(device.type == "cuda")
            ):

                outputs = model(
                    images
                )

                loss = criterion(
                    outputs,
                    labels
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            scaler.step(
                optimizer
            )

            scaler.update()


            running_loss += loss.item()


            predictions = outputs.argmax(
                dim=1
            )

            total += labels.size(0)

            correct += (
                predictions == labels
            ).sum().item()


        train_loss = (
            running_loss /
            len(train_loader)
        )

        train_acc = (
            correct /
            total
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_loss_total = 0.0

        val_correct = 0
        val_total = 0


        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(
                    device,
                    non_blocking=True
                )

                labels = labels.to(
                    device,
                    non_blocking=True
                )


                with torch.amp.autocast(
                    "cuda",
                    enabled=(device.type == "cuda")
                ):

                    outputs = model(
                        images
                    )

                    loss = criterion(
                        outputs,
                        labels
                    )


                val_loss_total += loss.item()


                predictions = outputs.argmax(
                    dim=1
                )


                val_total += labels.size(0)

                val_correct += (
                    predictions == labels
                ).sum().item()


        val_loss = (
            val_loss_total /
            len(val_loader)
        )

        val_acc = (
            val_correct /
            val_total
        )


        # ====================================================
        # LR
        # ====================================================

        lr = optimizer.param_groups[0]["lr"]


        # ====================================================
        # PRINT
        # ====================================================

        print(

            f"{variant} | "
            f"Epoch [{epoch+1:02d}/{EPOCHS}] "

            f"Train Loss: {train_loss:.4f} "
            f"Train Acc: {train_acc*100:.2f}% | "

            f"Val Loss: {val_loss:.4f} "
            f"Val Acc: {val_acc*100:.2f}% | "

            f"LR: {lr:.2e}"
        )


        # ====================================================
        # SAVE BEST
        # ====================================================

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            best_val_loss = val_loss

            best_epoch = epoch + 1

            best_train_acc = train_acc

            patience_counter = 0


            torch.save(

                {
                    "variant": variant,

                    "epoch": epoch + 1,

                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "val_acc":
                        val_acc,

                    "val_loss":
                        val_loss,

                    "train_acc":
                        train_acc,

                    "classes":
                        train_ds.classes
                },

                save_path
            )


            print(
                f"  >>> Best {variant} saved | "
                f"Val Acc: {val_acc*100:.2f}%"
            )


        else:

            patience_counter += 1


        # ====================================================
        # SCHEDULER
        # ====================================================

        scheduler.step()


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if patience_counter >= patience:

            print(
                f"  >>> Early stopping at "
                f"epoch {epoch+1}"
            )

            break


    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    checkpoint = torch.load(
        save_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()


    # ========================================================
    # EXACT FINAL ACCURACIES
    # ========================================================

    # Exact train evaluation
    exact_train_acc = evaluate_accuracy(model, train_loader)

    # Exact validation evaluation
    exact_val_acc = evaluate_accuracy(model, val_loader)

    # Exact test evaluation
    exact_test_acc = evaluate_accuracy(model, test_loader)

    # ========================================================
    # RESULT
    # ========================================================

    print("\n" + "-" * 70)

    print(f"{variant} FINAL RESULT")
    print("-" * 70)
    print(f"Best Epoch              : {best_epoch}")

    print(
        f"Exact Train Accuracy    : "
        f"{exact_train_acc:.4f} "
        f"({exact_train_acc*100:.2f}%)"
    )

    print(
        f"Exact Validation Acc.  : "
        f"{exact_val_acc:.4f} "
        f"({exact_val_acc*100:.2f}%)"
    )

    print(
        f"Exact Test Accuracy     : "
        f"{exact_test_acc:.4f} "
        f"({exact_test_acc*100:.2f}%)"
    )

    print("-" * 70)


    return {

        "Variant": variant,
        "Best Epoch": best_epoch,
        "Train Accuracy": exact_train_acc,
        "Validation Accuracy":exact_val_acc,

        "Test Accuracy":
            exact_test_acc
    }


# ============================================================
# RUN A3-A7
# ============================================================

all_results = []

for variant in VARIANTS:
    result = train_variant(variant)
    all_results.append(result)
    # Free GPU memory
    torch.cuda.empty_cache()


# ============================================================
# FINAL ABLATION TABLE
# ============================================================
results_df = pd.DataFrame(all_results)

print("\n\n")
print("=" * 80)
print("FINAL ABLATION STUDY RESULTS")
print("=" * 80)

display(
    results_df.style.format(
        {
            "Train Accuracy": "{:.4f}",
            "Validation Accuracy": "{:.4f}",
            "Test Accuracy": "{:.4f}"
        }
    )
)


# ============================================================
# PERCENTAGE TABLE
# ============================================================

percentage_df = results_df.copy()
percentage_df["Train Accuracy"] *= 100
percentage_df["Validation Accuracy"] *= 100
percentage_df["Test Accuracy"] *= 100

print("\n")
print("=" * 80)
print("ABLATION STUDY - PERCENTAGE")
print("=" * 80)

display(
    percentage_df.style.format(
        {
            "Train Accuracy" : "{:.2f}%",
            "Validation Accuracy" : "{:.2f}%",
            "Test Accuracy" : "{:.2f}%"
        }
    )
)


results_df.to_csv("/content/drive/MyDrive/Poultry/Results/ablation_A3_A7_results.csv", index=False)


print(
    "\nResults saved to: "
    "/content/drive/MyDrive/Poultry/Results/ablation_A3_A7_results.csv"
)